# PRACTICE 2: PREDICTING PRODUCT SALES — NHÓM 3
## NOTEBOOK DEMO: TRẢI NGHIỆM SUY LUẬN & DỰ ĐOÁN DOANH SỐ THỜI GIAN THỰC

> **Mục tiêu kỹ thuật:**
> - Nạp mô hình tối ưu đã lưu (`best_model.joblib`) và bộ tiền xử lý `preprocessor.joblib`.
> - Tái sử dụng 100% module từ `scripts/feature_engineering.py` (`prepare_sample_from_dict`).
> - Tích hợp cơ chế kiểm soát chất lượng đầu vào (Guardrails): bắt lỗi số lượng âm/rỗng hoặc dữ liệu thiếu logic.
> - Hỗ trợ so sánh trực quan đối chứng giữa các mô hình hồi quy Scratch.

### 1. Nạp Mô hình Tối ưu & Thiết lập Môi trường Suy luận
- Tự động nạp `best_model.joblib` và `metadata.json`.

In [ ]:
import os
import sys
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name in ['scripts', 'notebooks'] else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from scripts.feature_engineering import prepare_sample_from_dict

MODEL_PATH = BASE_DIR / 'models' / 'saved_models' / 'best_model.joblib'
META_PATH = BASE_DIR / 'models' / 'saved_models' / 'metadata.json'

assert MODEL_PATH.exists(), f"Chưa tìm thấy mô hình tại {MODEL_PATH}. Hãy chạy scripts/run_pipeline.py trước!"
model = joblib.load(MODEL_PATH)

with open(META_PATH, 'r', encoding='utf-8') as f:
    meta = json.load(f)

print(f"✓ Đã nạp thành công mô hình tối ưu: {meta.get('model_name')}")
print(f"✓ Thang đo kiểm thử: R2 = {meta.get('test_metrics', {}).get('R2 (Thực tế)')} | RMSE = {meta.get('test_metrics', {}).get('RMSE (INR)')} INR")

### 2. Xây dựng Hàm Suy luận Doanh số kèm Cơ chế Guardrails
- Kiểm tra hợp lệ: `Qty > 0`, kiểm tra các trường phân loại.
- Tự động trích xuất đặc trưng và đưa ra số tiền dự đoán (INR).

In [ ]:
def predict_product_sales(order_dict: dict, verbose: bool = True) -> float:
    """Hàm dự đoán doanh số đơn hàng thời gian thực kèm cơ chế phòng vệ Guardrail."""
    # 1. Cơ chế phòng vệ Guardrail
    qty = order_dict.get('Qty', 1)
    if qty is None or qty <= 0:
        if verbose:
            print(f"⚠️ [CẢNH BÁO GUARDRAIL]: Số lượng sản phẩm Qty={qty} không hợp lệ! Đơn hàng phải có Qty >= 1.")
        return 0.0

    # 2. Chuẩn bị đặc trưng đầu vào
    df_sample = prepare_sample_from_dict(order_dict)

    # 3. Suy luận qua Pipeline
    pred_log = model.predict(df_sample)[0]
    pred_inr = float(np.clip(np.expm1(pred_log), 0.0, None))

    if verbose:
        print("=" * 65)
        print("   📦 KẾT QUẢ SUY LUẬN DOANH SỐ ĐƠN HÀNG (INR)")
        print("=" * 65)
        print(f"• Ngành hàng (Category)    : {order_dict.get('Category')}")
        print(f"• Kích cỡ (Size)           : {order_dict.get('Size')}")
        print(f"• Số lượng đặt mua (Qty)   : {qty}")
        print(f"• Bang giao hàng           : {order_dict.get('ship-state', 'Other')}")
        print(f"• Kênh hoàn tất            : {order_dict.get('Fulfilment', 'Merchant')}")
        print(f"• Khách hàng B2B           : {'Có (Mua sỉ doanh nghiệp)' if order_dict.get('B2B') else 'Không (Bán lẻ cá nhân)'}")
        print(f"• Khuyến mãi áp dụng       : {order_dict.get('promotion-ids', 'No_Promotion')}")
        print("-" * 65)
        print(f"💰 DOANH SỐ DỰ BÁO        : {pred_inr:,.2f} INR (Rupee Ấn Độ)")
        print("=" * 65)

    return pred_inr

### 3. Thực nghiệm Các Tình huống Thực tế
- Ca 1: Khách lẻ mua 1 áo Kurta ngày thường không khuyến mãi.
- Ca 2: Khách sỉ B2B mua 5 bộ Set cao cấp cuối tuần có áp dụng khuyến mãi.
- Ca 3: Kiểm thử cơ chế phòng vệ Guardrail (nhập sai Qty = -2).

In [ ]:
print("--- THỰC NGHIỆM CA 1: KHÁCH LẺ MUA ÁO THỜI TRANG KURTA ---")
order_1 = {
    'Category': 'Kurta',
    'Size': 'M',
    'Qty': 1,
    'Fulfilment': 'Merchant',
    'ship-service-level': 'Standard',
    'Courier Status': 'Shipped',
    'ship-state': 'MAHARASHTRA',
    'Date': '04-30-22',
    'promotion-ids': 'No_Promotion',
    'B2B': False
}
predict_product_sales(order_1)

print("\n--- THỰC NGHIỆM CA 2: DOANH NGHIỆP B2B MUA SỈ 5 BỘ TRANG PHỤC SET ---")
order_2 = {
    'Category': 'Set',
    'Size': 'XL',
    'Qty': 5,
    'Fulfilment': 'Amazon',
    'ship-service-level': 'Expedited',
    'Courier Status': 'Shipped',
    'ship-state': 'KARNATAKA',
    'Date': '05-01-22',
    'promotion-ids': 'IN_PROMO_SUMMER_50',
    'B2B': True
}
predict_product_sales(order_2)

print("\n--- THỰC NGHIỆM CA 3: KIỂM THỬ PHÒNG VỆ GUARDRAIL (QTY KHÔNG HỢP LỆ) ---")
order_invalid = {
    'Category': 'Top',
    'Size': 'S',
    'Qty': -2,
    'ship-state': 'DELHI'
}
predict_product_sales(order_invalid)